# PlaceMux — Task 06: Quality Baseline Exploration

This notebook walks through the data and model interactively.  
For production runs, use `run_pipeline.py` instead.

**Run order:**
1. Generate data
2. Explore student / job distributions
3. Build features and inspect them
4. Evaluate baseline heuristic
5. Train ranker and compare

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# add project root to path
sys.path.insert(0, os.path.abspath('..'))

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Generate synthetic data

In [ ]:
from src.utils.data_generator import generate_students, generate_jobs, compute_ground_truth

students = generate_students(n=300)
jobs = generate_jobs(n=80)
labels = compute_ground_truth(students, jobs)

print(f'Students : {len(students)}')
print(f'Jobs     : {len(jobs)}')
print(f'Pairs    : {len(labels)}')
print(f'Positive rate: {labels["label"].mean():.1%}')

## 2. Explore skill score distributions

In [ ]:
from src.utils.data_generator import ALL_SKILLS

skill_means = students[ALL_SKILLS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
skill_means.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average verified skill score across all students', fontsize=13)
ax.set_ylabel('Mean score (0–100)')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/skill_score_distribution.png', dpi=120)
plt.show()

## 3. Label distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# overall distribution
labels['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Overall label distribution')
axes[0].set_xticklabels(['No Match (0)', 'Match (1)'], rotation=0)

# positive rate by job title
merged = labels.merge(jobs[['job_id', 'title']], on='job_id')
pos_by_role = merged.groupby('title')['label'].mean().sort_values()
pos_by_role.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Positive match rate by role')
axes[1].set_xlabel('Fraction of student–job pairs that are positive')

plt.tight_layout()
plt.savefig('../reports/label_distribution.png', dpi=120)
plt.show()

## 4. Build features and inspect

In [ ]:
from src.features.feature_engineering import build_pair_features

features = build_pair_features(students, jobs, labels)
print(features.shape)
features.head()

In [ ]:
# check feature distributions by label
feature_cols = ['skill_overlap_ratio', 'score_gap_mean', 'cgpa_gap', 'nice_to_have_overlap']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, feature_cols):
    for label_val, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        subset = features[features['label'] == label_val][col]
        subset.hist(bins=30, ax=ax, alpha=0.6, color=color,
                    label=f'label={label_val}')
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/feature_distributions.png', dpi=120)
plt.show()

## 5. Baseline heuristic evaluation

In [ ]:
from src.models.baseline import evaluate_baseline

baseline_metrics, scored_df = evaluate_baseline(features)
print('Baseline metrics:')
for k, v in baseline_metrics.items():
    print(f'  {k:<30} {v:.4f}')

## 6. Train ranker and compare

In [ ]:
# Note: run_pipeline.py or ranker.py must be run first to have train/val/test splits
import pandas as pd
import json

try:
    with open('../reports/ranker_metrics.json') as f:
        ranker_report = json.load(f)
    with open('../reports/baseline_metrics.json') as f:
        baseline_report = json.load(f)

    bm = baseline_report['test_metrics']
    rm = ranker_report['test_metrics']

    comparison = pd.DataFrame({
        'Baseline': bm,
        'RF Ranker': rm,
    }).loc[['precision', 'recall', 'f1', 'roc_auc', 'false_positive_rate', 'ndcg@10']]

    print(comparison.to_string())
except FileNotFoundError:
    print('Run run_pipeline.py first to generate model metrics.')

## 7. Feature importances
What does the model actually rely on?

In [ ]:
try:
    fi = pd.read_csv('../reports/feature_importances.csv')

    fig, ax = plt.subplots(figsize=(8, 5))
    fi.sort_values('importance').plot(
        kind='barh', x='feature', y='importance',
        ax=ax, legend=False, color='steelblue'
    )
    ax.set_title('Random Forest feature importances')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../reports/feature_importances.png', dpi=120)
    plt.show()
except FileNotFoundError:
    print('Run run_pipeline.py first.')